In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
path = "/data/pcpe/pcpe_03.csv"
df = pd.read_csv(path, delimiter=';')
df

/tmp/ipykernel_41768/2039808546.py:2: DtypeWarning: Columns (29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, delimiter=';')


,NUMERO_CASO,NUMERO_BANCO,NOME_BANCO,NUMERO_AGENCIA,NUMERO_CONTA,TIPO,CPF_CNPJ_TITULAR,NOME_TITULAR,DATA_LANCAMENTO,CPF_CNPJ_OD,...,VALOR_SALDO,NATUREZA_SALDO,NUMERO_BANCO_OD,NUMERO_AGENCIA_OD,NUMERO_CONTA_OD,NOME_ENDOSSANTE_CHEQUE,DOC_ENDOSSANTE_CHEQUE,DIA_LANCAMENTO,MES_LANCAMENTO,ANO_LANCAMENTO
0,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,2764,10081653,2,37469951830,MARTA VILLADICANI,2019-02-22 00:00:00,0,...,"128065,54",C,0,0,NaN,NaN,NaN,22,2,2019
1,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,2764,10081653,2,37469951830,MARTA VILLADICANI,2019-02-27 00:00:00,0,...,"131594,18",C,0,0,NaN,NaN,NaN,27,2,2019
2,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,2764,10081653,2,37469951830,MARTA VILLADICANI,2019-02-27 00:00:00,0,...,"131688,04",C,0,0,NaN,NaN,NaN,27,2,2019
3,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,2764,10081653,2,37469951830,MARTA VILLADICANI,2019-04-02 00:00:00,0,...,"135574,62",C,0,0,NaN,NaN,NaN,2,4,2019
4,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,2764,10081653,2,37469951830,MARTA VILLADICANI,2019-04-30 00:00:00,0,...,"135819,3",C,0,0,NaN,NaN,NaN,30,4,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167027,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,3929,5251117,1,6745923440,MORENA GALEATI,2022-10-31 00:00:00,43808741000106,...,"7066,6",C,403,1,4738204.0,NaN,NaN,31,10,2022
167028,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,3929,5251117,1,6745923440,MORENA GALEATI,2022-10-31 00:00:00,43808741000106,...,"5323,1",C,403,1,4738204.0,NaN,NaN,31,10,2022
167029,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,3929,5251117,1,6745923440,MORENA GALEATI,2022-10-31 00:00:00,10585973458,...,"9066,6",C,336,1,7924406.0,NaN,NaN,31,10,2022
167030,EXTRATO-2.0-F0R0B8,237,BANCO BRADESCO S.A,3929,5251117,1,6745923440,MORENA GALEATI,2022-10-31 00:00:00,10585973458,...,"13145,15",C,336,1,7924406.0,NaN,NaN,31,10,2022


In [5]:
tipologias = ["I-d", "I-e", "IV-n"]

# Dicionário para armazenar resultados
resumo = {
    "Accounts": [],
    "Individuals/Companies": [],
    "Transactions": []
}

colunas = []

for tip in tipologias:
    df_tip = df[df[tip] == 1]
    
    contas = df_tip["NUMERO_CONTA"].nunique()
    individuos = df_tip["CPF_CNPJ_TITULAR"].nunique()
    transacoes = len(df_tip)
    
    resumo["Accounts"].append(contas)
    resumo["Individuals/Companies"].append(individuos)
    resumo["Transactions"].append(transacoes)
    colunas.append(tip)

# "Nenhuma"
df_nenhuma = df[(df["I-d"] == 0) & (df["I-e"] == 0) & (df["IV-n"] == 0)]
resumo["Accounts"].append(df_nenhuma["NUMERO_CONTA"].nunique())
resumo["Individuals/Companies"].append(df_nenhuma["CPF_CNPJ_TITULAR"].nunique())
resumo["Transactions"].append(len(df_nenhuma))
colunas.append("None")

# Monta a tabela final
tabela = pd.DataFrame(resumo, index=colunas).T

print(tabela)


                         I-d   I-e  IV-n    None
Accounts                  40    35    12     167
Individuals/Companies     15    12     5      23
Transactions           19354  1362  7371  145906


In [6]:
def gerar_latex_tabela_corrigida(df, caption, label, header_names):
    """
    Formata o DataFrame para LaTeX, compatível com versões mais antigas do pandas
    e garante apenas \toprule, \midrule e \bottomrule.
    """
    # 1. Formatar os números
    df_formatado = df.astype(int).applymap(lambda x: f'{x:,}'.replace(',', '.'))

    # 2. Definir o formato das colunas e gerar a string LaTeX
    column_format = 'l' + 'r' * len(df.columns)
    
    latex_string = df_formatado.to_latex(
        index_names=False,
        column_format=column_format,
        header=False,
        escape=False 
    )

    # 3. Customizar cabeçalho e ambiente
    header_row = " & ".join(["Typology"] + header_names) + " \\\\"
    
    # Remove a primeira e a última linha (\begin{tabular} e \end{tabular})
    linhas = latex_string.splitlines()
    corpo_tabela = "\n".join(linhas[1:-1]).strip()

    # O template LaTeX JÁ ESTÁ CORRETO para apenas 1 \midrule e 1 \bottomrule
    latex_final = f"""
\\begin{{table}}[!htbp]
\\small
\\caption{{{caption}}}
\\centering
\\begin{{center}}
\\begin{{tabular}}{{{column_format}}}
\\toprule
{header_row}
\\midrule
{corpo_tabela}
\\bottomrule
\\end{{tabular}}
\\end{{center}}
\\label{{{label}}}
\\end{{table}}
"""
    return latex_final

# ------------------------------------------------------------
# --- GERAÇÃO FINAL DO CÓDIGO LaTeX ---
# ------------------------------------------------------------

caption = "Descriptive statistics by fraud typology on investigation 3."
label = "tab:summary_typologies"
# 'tabela.columns.tolist()' agora só contém as tipologias (sem Total)
nomes_cabecalho = tabela.columns.tolist() 

codigo_latex_final = gerar_latex_tabela_corrigida(
    tabela, 
    caption, 
    label, 
    nomes_cabecalho
)

print("\n\n#################################################")
print("### CÓDIGO LaTeX FINAL (Sem Total e Linhas Duplicadas) ###")
print("#################################################")
print(codigo_latex_final)
# ------------------------------------------------------------
# --- GERAÇÃO FINAL DO CÓDIGO LaTeX ---
# ------------------------------------------------------------

caption = "Descriptive statistics by fraud typology, including total count."
label = "tab:summary_typologies_total"
nomes_cabecalho = tabela.columns.tolist() 

codigo_latex_final = gerar_latex_tabela_corrigida(
    tabela, 
    caption, 
    label, 
    nomes_cabecalho
)

print("\n\n#################################################")
print("### CÓDIGO LaTeX PRONTO PARA OVERLEAF (Corrigido) ###")
print("#################################################")
print(codigo_latex_final)



#################################################
### CÓDIGO LaTeX FINAL (Sem Total e Linhas Duplicadas) ###
#################################################

\begin{table}[!htbp]
\small
\caption{Descriptive statistics by fraud typology on investigation 3.}
\centering
\begin{center}
\begin{tabular}{lrrrr}
\toprule
Typology & I-d & I-e & IV-n & None \\
\midrule
\toprule
\midrule
Accounts & 40 & 35 & 12 & 167 \\
Individuals/Companies & 15 & 12 & 5 & 23 \\
Transactions & 19.354 & 1.362 & 7.371 & 145.906 \\
\bottomrule
\bottomrule
\end{tabular}
\end{center}
\label{tab:summary_typologies}
\end{table}



#################################################
### CÓDIGO LaTeX PRONTO PARA OVERLEAF (Corrigido) ###
#################################################

\begin{table}[!htbp]
\small
\caption{Descriptive statistics by fraud typology, including total count.}
\centering
\begin{center}
\begin{tabular}{lrrrr}
\toprule
Typology & I-d & I-e & IV-n & None \\
\midrule
\toprule
\midrule
Accounts &

/tmp/ipykernel_41768/2529389736.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_formatado = df.astype(int).applymap(lambda x: f'{x:,}'.replace(',', '.'))
/tmp/ipykernel_41768/2529389736.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_formatado = df.astype(int).applymap(lambda x: f'{x:,}'.replace(',', '.'))
